# 19 — Transfer Learning and Pretrained Models

In the previous notebook, we learned how to debug PyTorch experiments and make them more reproducible.

Now we will study one of the most useful techniques in practical computer vision:

> **Transfer Learning**

Instead of training a CNN completely from random initialization, we begin with a model that has already learned useful visual features from a large dataset.

We can then adapt that model to our own task.

## In this notebook, we will study:

1. What is transfer learning?
2. Why pretrained models help
3. Feature extraction
4. Fine-tuning
5. Freezing layers
6. Replacing classifier heads
7. Pretrained CNNs in `torchvision`
8. Input normalization for pretrained models
9. Adapting RGB models to grayscale images
10. Training only the classifier
11. Unfreezing deeper layers
12. Different learning rates for different parameter groups
13. Saving fine-tuned models
14. Transfer learning for ultrasound images
15. Common transfer-learning mistakes
16. Debugging transfer-learning pipelines
17. Practice exercises

## Main Goal

By the end of this notebook, you should understand the two major transfer-learning strategies:

$$
\boxed{
\text{Feature Extraction}
=
\text{Freeze Backbone}
+
\text{Train New Head}
}
$$

and:

$$
\boxed{
\text{Fine-Tuning}
=
\text{Start Pretrained}
+
\text{Unfreeze Some Layers}
+
\text{Train Carefully}
}
$$

The most important principle is:

> **A pretrained model is not automatically compatible with your data.  
You must also match its expected input format, normalization, classifier output, and training strategy.**


In [ ]:
import copy
import torch
import torch.nn as nn
import torchvision

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)


# 1. What Is Transfer Learning?

Transfer learning means:

> Using knowledge learned on one task or dataset to help solve another task.

Instead of starting from completely random weights:

$$
\theta_0 \sim random
$$

we start from pretrained weights:

$$
\theta_0 = \theta_{pretrained}
$$

The pretrained network already contains visual filters that may detect patterns such as:

- Edges
- Corners
- Textures
- Shapes
- Object parts
- Higher-level visual structures

We then adapt those learned features to our target problem.


# 2. Training From Scratch vs Transfer Learning

## Training From Scratch

$$
\boxed{
Random\ Weights
\rightarrow
Train\ Entire\ Network
}
$$

## Transfer Learning

$$
\boxed{
Pretrained\ Weights
\rightarrow
Replace\ Head
\rightarrow
Adapt\ to\ New\ Task
}
$$

Transfer learning is especially attractive when:

- The target dataset is small
- Training resources are limited
- A strong pretrained architecture already exists
- The new task contains visual structure related to natural-image features


# 3. Why Pretrained Models Can Help

A CNN trained on a large image dataset learns reusable visual representations.

Earlier layers often learn fairly general features.

Later layers become increasingly specific to the original task.

A rough conceptual hierarchy is:

$$
\boxed{
\text{Edges}
\rightarrow
\text{Textures}
\rightarrow
\text{Shapes}
\rightarrow
\text{High-Level Semantics}
}
$$

This is why we can often keep much of the network and replace only the final classifier.


# 4. The Backbone and the Head

It is useful to split a classifier conceptually into:

$$
\boxed{
\text{Backbone}
+
\text{Classification Head}
}
$$

The **backbone** extracts features.

The **head** maps those features to task-specific outputs.

For a ResNet classifier:

- Convolutional body = backbone
- Final `fc` layer = classification head


# 5. Two Main Transfer-Learning Strategies

## Strategy A — Feature Extraction

Freeze most or all pretrained backbone parameters.

Train only the newly added classifier head.

$$
\boxed{
Frozen\ Backbone
\rightarrow
Trainable\ Head
}
$$

## Strategy B — Fine-Tuning

Start from pretrained weights, but allow some or all pretrained layers to update.

$$
\boxed{
Pretrained\ Backbone
\rightarrow
Small\ Controlled\ Updates
}
$$


# 6. When Is Feature Extraction Useful?

Feature extraction is often useful when:

- Dataset is small
- Target task is not extremely different
- You want a fast baseline
- You want to reduce overfitting risk
- Compute is limited

It is usually a strong first experiment.


# 7. When Is Fine-Tuning Useful?

Fine-tuning can help when:

- You have more data
- The target domain differs from the pretraining domain
- The frozen features are not sufficient
- You want higher task-specific performance

Fine-tuning requires more care because pretrained representations can be damaged by overly aggressive updates.


# 8. `torchvision` Pretrained Models

`torchvision.models` provides many standard CNN architectures.

Examples include:

- ResNet
- DenseNet
- EfficientNet
- MobileNet
- ConvNeXt
- Vision Transformer

In this notebook, we will use:

> **ResNet-18**

because it is small enough for teaching while still showing realistic transfer-learning patterns.


In [ ]:
print(models.resnet18)


# 9. Modern `torchvision` Weight API

Modern `torchvision` uses explicit weight enums.

For ResNet-18:

```python
models.ResNet18_Weights.DEFAULT
```

Then:

```python
model = models.resnet18(
    weights=weights
)
```

On the first run, pretrained weights may need to be downloaded.

If internet access is unavailable, `weights=None` creates the same architecture with random initialization.


In [ ]:
weights = models.ResNet18_Weights.DEFAULT

print(weights)


# 10. Loading a Pretrained ResNet-18

The following helper tries to load pretrained weights.

If weights cannot be downloaded in the current runtime, it falls back to random initialization so the notebook can still be explored.

For an actual transfer-learning experiment, verify that pretrained weights were loaded successfully.


In [ ]:
def load_resnet18_pretrained():
    weights = (
        models.ResNet18_Weights.DEFAULT
    )

    try:
        model = models.resnet18(
            weights=weights
        )

        pretrained_loaded = True

    except Exception as error:
        print(
            "Pretrained weights could not be loaded."
        )

        print(
            "Falling back to random initialization."
        )

        print(
            "Reason:",
            error
        )

        model = models.resnet18(
            weights=None
        )

        pretrained_loaded = False

    return (
        model,
        weights,
        pretrained_loaded
    )

model, weights, pretrained_loaded = (
    load_resnet18_pretrained()
)

print(
    "Pretrained loaded:",
    pretrained_loaded
)


# 11. Inspecting ResNet-18

Print the model before modifying anything.

This helps identify:

- First convolution
- Residual stages
- Pooling
- Final classifier


In [ ]:
print(model)


# 12. Finding the Final Classifier

For ResNet-18, the final classifier is:

```python
model.fc
```

It maps the final feature vector to the original pretraining classes.


In [ ]:
print(
    model.fc
)


# 13. Inspecting the Final Feature Dimension

Before replacing the classifier, read:

```python
model.fc.in_features
```

For ResNet-18, this is typically:

$$
512
$$

So the backbone produces a feature vector of size:

$$
512
$$

per image before the final classifier.


In [ ]:
num_features = (
    model.fc.in_features
)

print(
    "Final feature dimension:",
    num_features
)


# 14. Replacing the Classifier Head

Suppose our new problem has:

$$
3
$$

classes.

Replace:

```python
model.fc
```

with:

```python
nn.Linear(
    num_features,
    3
)
```


In [ ]:
num_classes = 3

model.fc = nn.Linear(
    num_features,
    num_classes
)

print(
    model.fc
)


# 15. Output Shape After Replacing the Head

For batch size:

$$
8
$$

and:

$$
3
$$

classes:

$$
\boxed{
output.shape=(8,\ 3)
}
$$

These are raw logits for `CrossEntropyLoss`.


In [ ]:
dummy_rgb = torch.randn(
    2,
    3,
    224,
    224
)

model.eval()

with torch.no_grad():
    logits = model(
        dummy_rgb
    )

print(
    "Output shape:",
    logits.shape
)


# 16. Input Requirements Matter

A pretrained model was trained using a specific preprocessing recipe.

You should not assume that arbitrary resizing or normalization is appropriate.

The weight enum provides the recommended preprocessing transform.


In [ ]:
preprocess = (
    weights.transforms()
)

print(
    preprocess
)


# 17. Pretrained Input Normalization

ImageNet-pretrained models commonly expect RGB images normalized approximately using:

$$
mean=
\begin{array}{|c|c|c|}
\hline
0.485 & 0.456 & 0.406 \\
\hline
\end{array}
$$

and:

$$
std=
\begin{array}{|c|c|c|}
\hline
0.229 & 0.224 & 0.225 \\
\hline
\end{array}
$$

However, instead of manually memorizing these values, prefer using the preprocessing associated with the selected pretrained weights when possible.


# 18. Why Pretrained Normalization Matters

The pretrained filters learned from inputs with a particular numerical distribution.

If you feed images with very different scaling, the activation distribution can shift.

That can reduce the usefulness of pretrained features.

So transfer learning requires matching:

- Image scale
- Channel format
- Resize/crop strategy
- Normalization


# 19. Feature Extraction — Freeze the Backbone

To freeze parameters:

```python
parameter.requires_grad = False
```

A common feature-extraction strategy is:

1. Load pretrained model
2. Freeze all existing parameters
3. Replace final classifier
4. Train only the new classifier


In [ ]:
feature_model, weights, loaded = (
    load_resnet18_pretrained()
)

for parameter in (
    feature_model.parameters()
):
    parameter.requires_grad = False

in_features = (
    feature_model.fc.in_features
)

feature_model.fc = nn.Linear(
    in_features,
    3
)

print(
    feature_model.fc
)


# 20. Why Replace the Head After Freezing?

If you freeze all parameters first and then create a new classifier layer, the new layer starts with:

```python
requires_grad=True
```

by default.

So the backbone remains frozen while the new head is trainable.


In [ ]:
for name, parameter in (
    feature_model.named_parameters()
):
    if parameter.requires_grad:
        print(
            "Trainable:",
            name,
            tuple(
                parameter.shape
            )
        )


# 21. Counting Frozen vs Trainable Parameters


In [ ]:
total_parameters = sum(
    p.numel()
    for p in feature_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in feature_model.parameters()
    if p.requires_grad
)

frozen_parameters = (
    total_parameters
    - trainable_parameters
)

print(
    "Total:",
    total_parameters
)

print(
    "Trainable:",
    trainable_parameters
)

print(
    "Frozen:",
    frozen_parameters
)


# 22. Optimizer Should Receive Trainable Parameters

A clean feature-extraction optimizer can use only parameters where:

```python
requires_grad=True
```


In [ ]:
optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        feature_model.parameters()
    ),
    lr=1e-3
)

print(
    optimizer
)


# 23. Feature Extraction Does Not Mean `model.eval()`

This distinction is important.

Freezing weights means:

```python
requires_grad=False
```

Training mode means:

```python
model.train()
```

These are different concepts.

A frozen model can still be placed in training mode.

Layers such as BatchNorm and Dropout may change behavior depending on mode.


# 24. BatchNorm Nuance During Feature Extraction

ResNet contains Batch Normalization layers.

Calling:

```python
model.train()
```

can update BatchNorm running statistics even if BatchNorm parameters are frozen.

There are different valid strategies depending on the experiment.

A common conservative feature-extraction strategy is to keep the frozen backbone's BatchNorm layers in evaluation behavior while training the new head.

This is an advanced detail worth remembering.


# 25. A Helper to Freeze BatchNorm Statistics

One possible helper:


In [ ]:
def set_frozen_batchnorm_eval(
    module
):
    if isinstance(
        module,
        nn.BatchNorm2d
    ):
        module.eval()

feature_model.train()

feature_model.apply(
    set_frozen_batchnorm_eval
)

print(
    "Frozen BatchNorm layers set to eval mode."
)


# 26. Training Only the Classifier

The training loop itself looks familiar:

```python
model.train()

for images, targets in loader:
    optimizer.zero_grad()
    logits = model(images)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer.step()
```

The difference is that frozen parameters do not receive normal gradient updates.


# 27. Creating a Small Fake RGB Dataset

To demonstrate the mechanics without downloading a real dataset, we can use:

`torchvision.datasets.FakeData`

This dataset is only for pipeline testing.

It does **not** represent a meaningful classification task.


In [ ]:
fake_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(
        (224, 224)
    ),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

fake_dataset = datasets.FakeData(
    size=64,
    image_size=(
        3,
        224,
        224
    ),
    num_classes=3,
    transform=fake_transform
)

fake_loader = DataLoader(
    fake_dataset,
    batch_size=8,
    shuffle=True
)

images, targets = next(
    iter(fake_loader)
)

print(
    "Images:",
    images.shape
)

print(
    "Targets:",
    targets.shape
)


# 28. One Feature-Extraction Training Step

Because FakeData labels are random, this step is only for verifying:

- Shapes
- Loss
- Gradients
- Trainable parameters


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

feature_model = feature_model.to(
    device
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        feature_model.parameters()
    ),
    lr=1e-3
)

feature_model.train()

feature_model.apply(
    set_frozen_batchnorm_eval
)

images = images.to(
    device
)

targets = targets.to(
    device
)

optimizer.zero_grad()

logits = feature_model(
    images
)

loss = criterion(
    logits,
    targets
)

loss.backward()

optimizer.step()

print(
    "Logits:",
    logits.shape
)

print(
    "Loss:",
    loss.item()
)


# 29. Verify Backbone Is Frozen

The newly replaced classifier should have gradients.

Frozen backbone parameters should not.


In [ ]:
for name, parameter in (
    feature_model.named_parameters()
):
    if (
        name.startswith(
            "fc."
        )
        or
        name.startswith(
            "layer4.1.conv2"
        )
    ):
        print(
            name,
            "| requires_grad:",
            parameter.requires_grad,
            "| grad is None:",
            parameter.grad is None
        )


# 30. Fine-Tuning

Feature extraction keeps pretrained backbone weights fixed.

Fine-tuning allows selected pretrained parameters to change.

A common strategy is:

1. Train the new classifier first
2. Unfreeze the final backbone stage
3. Continue training with a smaller learning rate


# 31. Why Fine-Tune Gradually?

The new classifier begins randomly initialized.

If you immediately update the entire pretrained network with a large learning rate, you may damage useful pretrained representations.

Gradual fine-tuning reduces this risk.


# 32. Unfreezing the Last ResNet Stage

ResNet's last major block is:

```python
model.layer4
```

We can unfreeze it:


In [ ]:
for parameter in (
    feature_model.layer4.parameters()
):
    parameter.requires_grad = True

trainable_count = sum(
    p.numel()
    for p in feature_model.parameters()
    if p.requires_grad
)

print(
    "Trainable parameters after unfreezing layer4:",
    trainable_count
)


# 33. Inspect Which Parameters Are Trainable


In [ ]:
for name, parameter in (
    feature_model.named_parameters()
):
    if parameter.requires_grad:
        print(
            name
        )


# 34. Different Learning Rates for Different Parameter Groups

The pretrained backbone often needs a smaller learning rate than the new classifier.

Example:

$$
\eta_{backbone}=10^{-4}
$$

$$
\eta_{head}=10^{-3}
$$

This allows cautious backbone adaptation while letting the new classifier learn faster.


In [ ]:
optimizer = torch.optim.AdamW([
    {
        "params":
            feature_model.layer4.parameters(),
        "lr":
            1e-4
    },
    {
        "params":
            feature_model.fc.parameters(),
        "lr":
            1e-3
    }
],
weight_decay=1e-4)

for index, group in enumerate(
    optimizer.param_groups
):
    print(
        f"Group {index}:",
        "lr =",
        group["lr"],
        "| tensors =",
        len(
            group["params"]
        )
    )


# 35. Fine-Tuning More Layers

You can progressively unfreeze:

- `layer4`
- then `layer3`
- then more of the backbone

But more unfreezing means:

- More trainable parameters
- More memory use
- More overfitting risk
- Greater chance of disturbing pretrained features

Unfreeze based on validation evidence, not habit.


# 36. Full Fine-Tuning

Full fine-tuning means:

```python
for parameter in model.parameters():
    parameter.requires_grad = True
```

This can work well when:

- You have enough target-domain data
- The domain differs substantially
- Learning rate is carefully controlled


In [ ]:
full_tune_model, _, _ = (
    load_resnet18_pretrained()
)

full_tune_model.fc = nn.Linear(
    full_tune_model.fc.in_features,
    3
)

for parameter in (
    full_tune_model.parameters()
):
    parameter.requires_grad = True

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in full_tune_model.parameters()
        if p.requires_grad
    )
)


# 37. Fine-Tuning Learning Rate

Pretrained layers often use a smaller learning rate than randomly initialized layers.

Why?

Pretrained weights are already useful.

We usually want:

$$
\text{small adaptation}
$$

rather than:

$$
\text{large random movement}
$$


# 38. Input Normalization for Pretrained Models

A pretrained model is coupled to its preprocessing.

The safest pattern is:

```python
weights = ...
transform = weights.transforms()
```

This ensures preprocessing matches the selected weight recipe.


In [ ]:
weights = models.ResNet18_Weights.DEFAULT

recommended_transform = (
    weights.transforms()
)

print(
    recommended_transform
)


# 39. Training Augmentation With Pretrained Normalization

The pretrained transform is often designed for evaluation/inference.

For training, you may want random augmentation before normalization.

A common ImageNet-style training pipeline is conceptually:

```python
RandomResizedCrop
RandomHorizontalFlip
ToTensor
Normalize
```

The exact augmentation should still match your domain.


In [ ]:
imagenet_mean = [
    0.485,
    0.456,
    0.406
]

imagenet_std = [
    0.229,
    0.224,
    0.225
]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])

val_transform = transforms.Compose([
    transforms.Resize(
        256
    ),
    transforms.CenterCrop(
        224
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])

print(
    train_transform
)


# 40. Do Not Copy Natural-Image Augmentation Blindly

For ultrasound, some ImageNet-style transforms may be inappropriate.

Ask whether a transform preserves:

- Anatomy
- Orientation
- Clinical interpretation
- Pathology label

For example, a horizontal flip may be valid for one ultrasound task and invalid for another.


# 41. The Grayscale Problem

Many pretrained CNNs expect:

$$
3
$$

input channels because they were pretrained on RGB images.

Ultrasound images may naturally be:

$$
1
$$

channel.

There are several strategies for adapting this mismatch.


# 42. Strategy A — Repeat Grayscale Into Three Channels

A simple approach is:

$$
(1,\ H,\ W)
\rightarrow
(3,\ H,\ W)
$$

by copying the grayscale channel three times.

The pretrained first convolution remains unchanged.


In [ ]:
gray = torch.randn(
    1,
    224,
    224
)

rgb_like = gray.repeat(
    3,
    1,
    1
)

print(
    "Gray:",
    gray.shape
)

print(
    "Repeated:",
    rgb_like.shape
)


# 43. `transforms.Grayscale(num_output_channels=3)`

For PIL or compatible tensor pipelines, you can request a 3-channel grayscale representation:

```python
transforms.Grayscale(
    num_output_channels=3
)
```

This keeps the pretrained RGB input interface intact.


In [ ]:
grayscale_to_rgb = transforms.Grayscale(
    num_output_channels=3
)

print(
    grayscale_to_rgb
)


# 44. Advantages of Repeating the Channel

Advantages:

- Keeps pretrained first convolution unchanged
- Simple
- Easy to implement
- Preserves compatibility with pretrained normalization

Disadvantage:

- The three channels contain identical information


# 45. Strategy B — Modify the First Convolution

ResNet-18 begins with:

```python
model.conv1
```

which expects:

$$
3
$$

channels.

We can replace it with a new convolution that expects:

$$
1
$$

channel.


In [ ]:
gray_model, _, gray_loaded = (
    load_resnet18_pretrained()
)

print(
    gray_model.conv1
)


# 46. Replacing `conv1` for One Channel

We want the same:

- Output channels
- Kernel size
- Stride
- Padding

but change:

$$
in\_channels:
3\rightarrow1
$$


In [ ]:
old_conv = gray_model.conv1

new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

gray_model.conv1 = new_conv

print(
    gray_model.conv1
)


# 47. Problem With Randomly Replacing the First Convolution

If we replace `conv1` with a fresh layer, its weights are randomly initialized.

We lose the pretrained information in the first convolution.

A better option is to initialize the one-channel kernel from the pretrained RGB kernels.


# 48. Averaging RGB Kernels Into One Channel

Original first-layer weight shape:

$$
(64,\ 3,\ 7,\ 7)
$$

Average across RGB channels:

$$
mean_{channel}
$$

to obtain:

$$
\boxed{
(64,\ 1,\ 7,\ 7)
}
$$


In [ ]:
rgb_model, _, rgb_loaded = (
    load_resnet18_pretrained()
)

old_weight = (
    rgb_model.conv1.weight
    .detach()
    .clone()
)

print(
    "Original:",
    old_weight.shape
)

gray_weight = old_weight.mean(
    dim=1,
    keepdim=True
)

print(
    "Averaged:",
    gray_weight.shape
)


# 49. Building a One-Channel ResNet Using Averaged Weights


In [ ]:
def make_grayscale_resnet18(
    num_classes,
    use_pretrained=True
):
    if use_pretrained:
        model, weights, loaded = (
            load_resnet18_pretrained()
        )
    else:
        model = models.resnet18(
            weights=None
        )

        weights = None
        loaded = False

    old_conv = model.conv1

    new_conv = nn.Conv2d(
        in_channels=1,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False
    )

    if loaded:
        with torch.no_grad():
            new_conv.weight.copy_(
                old_conv.weight.mean(
                    dim=1,
                    keepdim=True
                )
            )

    model.conv1 = new_conv

    model.fc = nn.Linear(
        model.fc.in_features,
        num_classes
    )

    return (
        model,
        weights,
        loaded
    )

gray_model, _, loaded = (
    make_grayscale_resnet18(
        num_classes=3
    )
)

print(
    gray_model.conv1
)

print(
    gray_model.fc
)


# 50. Verifying Grayscale Input

Now the model accepts:

$$
(N,\ 1,\ H,\ W)
$$


In [ ]:
gray_batch = torch.randn(
    2,
    1,
    224,
    224
)

gray_model.eval()

with torch.no_grad():
    gray_logits = gray_model(
        gray_batch
    )

print(
    "Output:",
    gray_logits.shape
)


# 51. Which Grayscale Strategy Should You Use?

There is no universal answer.

## Repeat to 3 Channels

Pros:

- Maximum compatibility with pretrained model
- First layer unchanged
- Easy baseline

## Modify First Convolution

Pros:

- Native one-channel input
- Slightly fewer parameters
- More natural interface for grayscale data

Cons:

- Requires deciding how to initialize adapted first-layer weights

A good experiment can compare both using the same validation split.


# 52. Ultrasound Normalization With Pretrained Models

This is an important research choice.

Possible strategies include:

### Option A

Convert ultrasound to 3 channels and use ImageNet normalization.

### Option B

Convert to 3 channels but use training-derived ultrasound statistics.

### Option C

Use a 1-channel first convolution and training-derived grayscale normalization.

Different approaches may work differently.

The choice should be documented and validated experimentally.


# 53. Why Domain Shift Matters

ImageNet contains natural photographs.

Ultrasound has very different characteristics:

- Speckle
- Acoustic shadows
- Machine-specific processing
- Probe-dependent appearance
- Grayscale intensity distributions

So pretrained filters may still help, but the domain gap can make fine-tuning more important.


# 54. Transfer Learning Does Not Remove the Need for Good Splits

Even a strong pretrained model can exploit leakage.

For ultrasound, still check:

- Patient-level split
- Study-level split
- Device/site effects
- Duplicate frames
- Text overlays
- Class imbalance

Transfer learning improves optimization, not experimental validity.


# 55. Training Only the Classifier Head

A common first-stage protocol:

1. Load pretrained model
2. Freeze backbone
3. Replace classifier
4. Train classifier for several epochs
5. Save best validation checkpoint


# 56. Reusable Freeze Helper


In [ ]:
def freeze_all(
    model
):
    for parameter in (
        model.parameters()
    ):
        parameter.requires_grad = False

def unfreeze_module(
    module
):
    for parameter in (
        module.parameters()
    ):
        parameter.requires_grad = True


# 57. Reusable ResNet Feature-Extraction Builder


In [ ]:
def build_resnet18_feature_extractor(
    num_classes
):
    model, weights, loaded = (
        load_resnet18_pretrained()
    )

    freeze_all(
        model
    )

    in_features = (
        model.fc.in_features
    )

    model.fc = nn.Linear(
        in_features,
        num_classes
    )

    return (
        model,
        weights,
        loaded
    )

feature_model, weights, loaded = (
    build_resnet18_feature_extractor(
        num_classes=3
    )
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in feature_model.parameters()
        if p.requires_grad
    )
)


# 58. A Clean Training Function

The transfer-learning training loop is still ordinary PyTorch training.


In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    freeze_batchnorm=False
):
    model.train()

    if freeze_batchnorm:
        model.apply(
            set_frozen_batchnorm_eval
        )

    loss_sum = 0.0
    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad()

        logits = model(
            images
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()

        optimizer.step()

        batch_size = (
            targets.size(0)
        )

        loss_sum += (
            loss.item()
            * batch_size
        )

        correct += (
            logits.argmax(
                dim=1
            )
            == targets
        ).sum().item()

        total += batch_size

    return (
        loss_sum / total,
        correct / total
    )


# 59. Validation Function


In [ ]:
def evaluate(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    loss_sum = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(
                device
            )

            targets = targets.to(
                device
            )

            logits = model(
                images
            )

            loss = criterion(
                logits,
                targets
            )

            batch_size = (
                targets.size(0)
            )

            loss_sum += (
                loss.item()
                * batch_size
            )

            correct += (
                logits.argmax(
                    dim=1
                )
                == targets
            ).sum().item()

            total += batch_size

    return (
        loss_sum / total,
        correct / total
    )


# 60. Best-Checkpoint Training Skeleton


In [ ]:
def fit_transfer_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=5,
    freeze_batchnorm=False
):
    best_val_loss = float(
        "inf"
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(
        epochs
    ):
        train_loss, train_acc = (
            train_one_epoch(
                model,
                train_loader,
                criterion,
                optimizer,
                device,
                freeze_batchnorm=(
                    freeze_batchnorm
                )
            )
        )

        val_loss, val_acc = (
            evaluate(
                model,
                val_loader,
                criterion,
                device
            )
        )

        history["train_loss"].append(
            train_loss
        )

        history["train_acc"].append(
            train_acc
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_acc"].append(
            val_acc
        )

        if val_loss < best_val_loss:
            best_val_loss = (
                val_loss
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train Loss {train_loss:.4f} | "
            f"Train Acc {train_acc:.3f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Acc {val_acc:.3f}"
        )

    model.load_state_dict(
        best_state
    )

    return history


# 61. Progressive Fine-Tuning Strategy

A practical staged strategy can be:

## Stage 1

Train only classifier head.

## Stage 2

Unfreeze final backbone block.

## Stage 3

Optionally unfreeze more layers.

At each stage:

- Use validation metrics
- Reduce learning rate for pretrained layers
- Keep best checkpoint


# 62. Why Different Learning Rates Help

Suppose the new head is random and the backbone is pretrained.

The head needs large adaptation:

$$
\Delta\theta_{head}
$$

The backbone usually needs smaller adaptation:

$$
\Delta\theta_{backbone}
$$

So parameter groups can use different learning rates.


# 63. Fine-Tuning Parameter Groups Example


In [ ]:
model, _, _ = (
    build_resnet18_feature_extractor(
        num_classes=3
    )
)

unfreeze_module(
    model.layer4
)

optimizer = torch.optim.AdamW([
    {
        "params":
            model.layer4.parameters(),
        "lr":
            1e-4
    },
    {
        "params":
            model.fc.parameters(),
        "lr":
            1e-3
    }
],
weight_decay=1e-4)

for index, group in enumerate(
    optimizer.param_groups
):
    print(
        "Group",
        index,
        "LR:",
        group["lr"]
    )


# 64. Saving a Fine-Tuned Model

Save the learned weights:

```python
torch.save(
    model.state_dict(),
    path
)
```

Also save:

- Architecture name
- Number of classes
- Grayscale strategy
- Pretrained weight version
- Input size
- Normalization
- Class mapping
- Fine-tuning policy


In [ ]:
checkpoint_metadata = {
    "architecture":
        "resnet18",

    "num_classes":
        3,

    "pretrained_weights":
        str(
            models.ResNet18_Weights.DEFAULT
        ),

    "input_size":
        224,

    "grayscale_strategy":
        "repeat_to_3_channels",

    "fine_tuning":
        "layer4 + classifier"
}

print(
    checkpoint_metadata
)


# 65. Saving a Training Checkpoint

If you want to resume training, save more than model weights.


In [ ]:
checkpoint = {
    "model_state_dict":
        model.state_dict(),

    "optimizer_state_dict":
        optimizer.state_dict(),

    "metadata":
        checkpoint_metadata
}

torch.save(
    checkpoint,
    "resnet18_transfer_checkpoint.pth"
)

print(
    "Checkpoint saved."
)


# 66. Loading a Fine-Tuned Checkpoint

The architecture must match the saved state.


In [ ]:
loaded_model, _, _ = (
    build_resnet18_feature_extractor(
        num_classes=3
    )
)

unfreeze_module(
    loaded_model.layer4
)

loaded_optimizer = (
    torch.optim.AdamW([
        {
            "params":
                loaded_model.layer4.parameters(),
            "lr":
                1e-4
        },
        {
            "params":
                loaded_model.fc.parameters(),
            "lr":
                1e-3
        }
    ],
    weight_decay=1e-4)
)

loaded_checkpoint = torch.load(
    "resnet18_transfer_checkpoint.pth",
    map_location="cpu",
    weights_only=False
)

loaded_model.load_state_dict(
    loaded_checkpoint[
        "model_state_dict"
    ]
)

loaded_optimizer.load_state_dict(
    loaded_checkpoint[
        "optimizer_state_dict"
    ]
)

print(
    loaded_checkpoint[
        "metadata"
    ]
)


# 67. Common Mistake — Forgetting to Replace the Final Head

A pretrained model may output:

$$
1000
$$

ImageNet logits.

If your task has:

$$
3
$$

classes, your final output should usually be:

$$
(batch,\ 3)
$$

Always inspect:

```python
model.fc
```

or the architecture-specific classifier.


# 68. Common Mistake — Wrong Preprocessing

A pretrained model may perform poorly if you use:

- Wrong image scale
- Wrong resize
- Wrong normalization
- Wrong channel order

Pretrained weights and preprocessing belong together.


# 69. Common Mistake — Updating the Entire Backbone Accidentally

If you intended feature extraction but forgot to freeze parameters, you are fine-tuning the whole model.

Always count trainable parameters before training.


In [ ]:
def count_trainable_parameters(
    model
):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

print(
    "Trainable:",
    count_trainable_parameters(
        feature_model
    )
)


# 70. Common Mistake — Freezing the New Head Too

If you freeze the model **after** replacing the classifier:

```python
model.fc = new_head

for p in model.parameters():
    p.requires_grad = False
```

you also freeze the new head.

Then nothing learns.

Always verify which parameters remain trainable.


# 71. Common Mistake — Learning Rate Too Large for Fine-Tuning

A high learning rate can rapidly overwrite useful pretrained representations.

Symptoms may include:

- Validation performance collapses
- Loss spikes
- Pretrained features degrade quickly

Fine-tuned backbone layers often benefit from smaller learning rates.


# 72. Common Mistake — Not Rebuilding the Optimizer After Changing Trainable Layers

Suppose you first train only the head.

Then you unfreeze `layer4`.

Your optimizer should now include the newly trainable parameters.

A clean approach is to create a new optimizer after changing the trainable set.


# 73. Common Mistake — Assuming `requires_grad=False` Freezes BatchNorm State

Freezing parameter gradients does not automatically stop BatchNorm running-statistic updates.

Remember the distinction between:

- Parameter gradients
- Module train/eval mode


# 74. Common Mistake — Converting Grayscale Incorrectly

If you use a pretrained RGB model, decide deliberately between:

- Repeating grayscale to 3 channels
- Replacing first convolution

Do not accidentally feed:

$$
1
$$

channel to a layer expecting:

$$
3
$$

channels.


# 75. Common Mistake — Using ImageNet Augmentation Without Domain Validation

Natural-image augmentation is not automatically appropriate for ultrasound.

For example:

- Flips
- Large rotations
- Aggressive crops
- Color jitter

may change clinical meaning.

Use domain-valid augmentation.


# 76. Common Mistake — Comparing Scratch and Pretrained Models Unfairly

A fair comparison should keep as much as possible fixed:

- Data split
- Input resolution
- Training epochs
- Evaluation metric
- Augmentation
- Class mapping
- Seed policy

Then vary the initialization/training strategy.


# 77. Common Mistake — Using Test Performance to Decide How Much to Unfreeze

Use validation data to choose:

- Frozen vs fine-tuned
- Which layers to unfreeze
- Learning rates
- Number of epochs

Keep test data untouched until final evaluation.


# 78. Transfer Learning for Ultrasound — Recommended Baseline Experiments

A strong experimental sequence is:

## Baseline A

Small CNN trained from scratch.

## Baseline B

Pretrained ResNet with grayscale repeated to 3 channels, frozen backbone.

## Baseline C

Same pretrained ResNet with final stage fine-tuned.

## Baseline D

One-channel adapted ResNet with averaged first-layer kernels.

Compare all models on the exact same patient-level validation split.


# 79. What Should You Record for Ultrasound Transfer Learning?

Record:

- Backbone architecture
- Pretrained weight source
- Image size
- Channel strategy
- Normalization
- Augmentation
- Patient-level split
- Frozen layers
- Fine-tuned layers
- Learning rates
- Weight decay
- Epochs
- Best checkpoint metric
- Seed
- Device/site distribution


# 80. Transfer-Learning Debugging Checklist

Before training:

1. Did pretrained weights load?
2. Is the input channel count correct?
3. Is preprocessing compatible?
4. Is the classifier output dimension correct?
5. Which parameters are trainable?
6. Are BatchNorm layers behaving as intended?
7. Does the optimizer contain the intended parameters?
8. Are learning rates appropriate?
9. Does one forward pass work?
10. Does one backward pass work?
11. Do trainable parameters change?
12. Does validation use `model.eval()`?


# 81. Inspecting Trainable Parameters


In [ ]:
for name, parameter in (
    model.named_parameters()
):
    if parameter.requires_grad:
        print(
            name,
            tuple(
                parameter.shape
            )
        )


# 82. Checking Optimizer Coverage

A useful sanity check is to compare the IDs of trainable parameters with the IDs contained in optimizer parameter groups.


In [ ]:
trainable_ids = {
    id(parameter)
    for parameter in model.parameters()
    if parameter.requires_grad
}

optimizer_ids = {
    id(parameter)
    for group in optimizer.param_groups
    for parameter in group["params"]
}

print(
    "All trainable parameters are in optimizer:",
    trainable_ids.issubset(
        optimizer_ids
    )
)


# 83. Checking One Fine-Tuning Gradient

After a backward pass:

- New classifier should have gradients
- Unfrozen backbone stage should have gradients
- Frozen earlier backbone should not


In [ ]:
model = model.to(
    device
)

criterion = nn.CrossEntropyLoss()

images, targets = next(
    iter(fake_loader)
)

images = images.to(
    device
)

targets = targets.to(
    device
)

optimizer = torch.optim.AdamW([
    {
        "params":
            model.layer4.parameters(),
        "lr":
            1e-4
    },
    {
        "params":
            model.fc.parameters(),
        "lr":
            1e-3
    }
])

optimizer.zero_grad()

loss = criterion(
    model(images),
    targets
)

loss.backward()

print(
    "fc grad:",
    model.fc.weight.grad is not None
)

print(
    "layer4 grad:",
    model.layer4[0].conv1.weight.grad is not None
)

print(
    "layer1 grad:",
    model.layer1[0].conv1.weight.grad is not None
)


# 84. Feature Extraction vs Fine-Tuning Summary

$$
\begin{array}{|c|c|c|}
\hline
\textbf{Property} & \textbf{Feature Extraction} & \textbf{Fine-Tuning} \\
\hline
Backbone & Frozen & Partly/fully\ trainable \\
\hline
Trainable\ params & Few & More \\
\hline
Memory & Lower & Higher \\
\hline
Overfitting\ risk & Lower & Higher \\
\hline
Adaptation & Limited & Stronger \\
\hline
Typical\ LR & Head\ LR & Smaller\ backbone\ LR \\
\hline
\end{array}
$$


# 85. Practice Exercises

Try these before looking at the solutions.

## Exercise 1

Load a ResNet-18 architecture.

## Exercise 2

Inspect the input features of `model.fc`.

## Exercise 3

Replace the classifier for 5 classes.

## Exercise 4

Freeze all backbone parameters and verify only the new head is trainable.

## Exercise 5

Count total and trainable parameters.

## Exercise 6

Unfreeze `layer4`.

## Exercise 7

Create two optimizer parameter groups:

- `layer4` at `1e-4`
- classifier at `1e-3`

## Exercise 8

Convert a one-channel image tensor into three identical channels.

## Exercise 9

Replace ResNet-18's first convolution so it accepts one channel.

## Exercise 10

Initialize the one-channel convolution using the mean of pretrained RGB kernels.


# 86. Conceptual Challenges

## Challenge 1

Why can pretrained CNN features help a new task?

## Challenge 2

What is the difference between feature extraction and fine-tuning?

## Challenge 3

Why should the final classifier be replaced?

## Challenge 4

Why can fine-tuning use a smaller backbone learning rate?

## Challenge 5

Why does freezing parameters not automatically freeze BatchNorm running statistics?

## Challenge 6

Why can pretrained normalization matter?

## Challenge 7

What are two reasonable ways to use RGB-pretrained models with grayscale ultrasound?

## Challenge 8

Why might full fine-tuning help more when the target domain differs strongly from ImageNet?

## Challenge 9

Why should patient-level splitting still be used with transfer learning?

## Challenge 10

Why should test performance not determine which layers are unfrozen?


# 87. Exercise Solutions


In [ ]:
# Exercise 1
exercise_model = models.resnet18(
    weights=None
)

# Exercise 2
exercise_in_features = (
    exercise_model.fc.in_features
)

print(
    "Exercise 2:",
    exercise_in_features
)

# Exercise 3
exercise_model.fc = nn.Linear(
    exercise_in_features,
    5
)

print(
    "Exercise 3:",
    exercise_model.fc
)

# Exercise 4
for parameter in (
    exercise_model.parameters()
):
    parameter.requires_grad = False

exercise_model.fc = nn.Linear(
    exercise_in_features,
    5
)

print(
    "Exercise 4 trainable:",
    [
        name
        for name, parameter
        in exercise_model.named_parameters()
        if parameter.requires_grad
    ]
)

# Exercise 5
print(
    "Total:",
    sum(
        p.numel()
        for p in exercise_model.parameters()
    )
)

print(
    "Trainable:",
    sum(
        p.numel()
        for p in exercise_model.parameters()
        if p.requires_grad
    )
)


In [ ]:
# Exercise 6
for parameter in (
    exercise_model.layer4.parameters()
):
    parameter.requires_grad = True

# Exercise 7
exercise_optimizer = torch.optim.AdamW([
    {
        "params":
            exercise_model.layer4.parameters(),
        "lr":
            1e-4
    },
    {
        "params":
            exercise_model.fc.parameters(),
        "lr":
            1e-3
    }
])

print(
    "Exercise 7 groups:",
    len(
        exercise_optimizer.param_groups
    )
)

# Exercise 8
gray = torch.randn(
    1,
    224,
    224
)

three_channel = gray.repeat(
    3,
    1,
    1
)

print(
    "Exercise 8:",
    three_channel.shape
)


In [ ]:
# Exercise 9 and 10
exercise_rgb_model = models.resnet18(
    weights=None
)

old_conv = (
    exercise_rgb_model.conv1
)

old_weight = (
    old_conv.weight
    .detach()
    .clone()
)

new_conv = nn.Conv2d(
    1,
    old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

with torch.no_grad():
    new_conv.weight.copy_(
        old_weight.mean(
            dim=1,
            keepdim=True
        )
    )

exercise_rgb_model.conv1 = (
    new_conv
)

print(
    "Exercise 9/10:",
    exercise_rgb_model.conv1.weight.shape
)


# 88. Key Takeaways

In this notebook, we learned:

- What transfer learning is
- Why pretrained models can help
- Backbone vs classifier head
- Feature extraction
- Fine-tuning
- Freezing parameters
- Replacing classifier heads
- Pretrained `torchvision` models
- Modern weight enums
- Pretrained input normalization
- Training only the classifier
- BatchNorm nuance
- Progressive unfreezing
- Different learning rates for parameter groups
- Saving fine-tuned models
- Loading checkpoints
- RGB-to-grayscale adaptation
- Repeating grayscale to 3 channels
- Replacing the first convolution
- Averaging pretrained RGB kernels
- Transfer learning for ultrasound
- Domain shift
- Common transfer-learning mistakes
- Transfer-learning debugging

The most important patterns are:

## Feature Extraction

$$
\boxed{
Pretrained\ Backbone
\rightarrow
Freeze
\rightarrow
Replace\ Head
\rightarrow
Train\ Head
}
$$

## Fine-Tuning

$$
\boxed{
Pretrained\ Model
\rightarrow
Train\ Head
\rightarrow
Unfreeze\ Selected\ Layers
\rightarrow
Use\ Smaller\ LR
}
$$

## Grayscale Ultrasound

Two useful baselines are:

$$
\boxed{
1\ channel
\rightarrow
repeat\ to\ 3
\rightarrow
RGB\ pretrained\ model
}
$$

or:

$$
\boxed{
modify\ first\ conv:
3\rightarrow1
}
$$


# 89. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What is transfer learning?
2. Why can pretrained weights help?
3. What is a backbone?
4. What is a classifier head?
5. What is feature extraction?
6. What is fine-tuning?
7. How do you freeze a parameter?
8. Why must the classifier head be replaced?
9. How do you inspect ResNet's final feature dimension?
10. Why is preprocessing tied to pretrained weights?
11. Why should raw logits still be used with `CrossEntropyLoss`?
12. Why can BatchNorm remain a concern even when parameters are frozen?
13. Why might you train only the classifier first?
14. What does progressive unfreezing mean?
15. Why use different learning rates for backbone and head?
16. What is full fine-tuning?
17. How can a grayscale image be adapted to an RGB-pretrained network?
18. How can a pretrained first convolution be adapted from 3 channels to 1?
19. Why might transfer learning still help ultrasound despite domain shift?
20. Why does transfer learning not eliminate the need for patient-level splitting?
21. Why should domain-valid augmentation still be used?
22. Why must the optimizer be rebuilt or updated after unfreezing layers?
23. Why should test data not guide the fine-tuning strategy?
24. What should be stored with a fine-tuned checkpoint?
25. What would you compare when deciding between scratch training and transfer learning?


# Next Notebook

# 20 — Advanced PyTorch Training Techniques

In the next notebook, we will study:

- Learning-rate schedulers
- `StepLR`
- `ReduceLROnPlateau`
- Cosine annealing
- Warmup intuition
- Mixed-precision training
- `torch.autocast`
- Gradient scaling
- Gradient accumulation
- Larger effective batch sizes
- Gradient clipping in real loops
- Efficient inference
- `torch.inference_mode()`
- Checkpoint resuming
- Training-loop instrumentation
- Practical performance optimization
